# Overview

We are now going to investigate how to calculate the Gibbs free energy of a periodic solid. This is a bit trickier mathematically, but the same process needs to take place: we must optimize the structure, get the energy, and then calculate the vibrational modes. In the case of solids, these vibrational modes are called phonons.

While ASE has builtin routines for calculating phonon modes of solids and the thermochemistry of solids (see `CrystalThermo`), there is a more powerful code called [phonopy](https://github.com/phonopy/phonopy) that is generally adopted in the community. Phonopy is a bit complicated to use, so we will rely on a package called [matcalc](https://github.com/materialyzeai/matcalc) that makes it a lot easier.

We will use `matcalc` extensively when we discuss machine learning interatomic potentials in greater detial.


# Setup


In [ ]:
!uv pip install "git+https://github.com/materialyzeai/matcalc.git" "upet>=0.2.1"

We will use the UPET machine learning potential here since it's reasonably accurate and fast. It can be thought of as a drop-in replacement for DFT. We need to download the model checkpoint.


In [ ]:
from upet.calculator import UPETCalculator

calc = UPETCalculator(model="pet-mad-s", version="1.5.0")

# Demonstration

We will calculate the Gibbs free energy of a 2x2x2 unit cell of Cu.


## Matcalc Overview


In [ ]:
from ase.build import bulk

atoms = bulk("Cu") * (2, 2, 2)

First, let me introduce you to `matcalc` briefly. `matcalc` is basically a wrapper around ASE and some other utilities. You know those relaxations you have been carrying out in ASE? They can be carried out using `matcalc` too. Let's try that out.


In [ ]:
from matcalc import RelaxCalc

relax = RelaxCalc(calc, optimizer="BFGS", fmax=0.01, relax_cell=True)

From the above example, it should hopefully be clear that this is basically just a streamlined interface to ASE. The benefit is somebody has taken care of figuring out what to do for you. The downside is means you have less control, and you must be very careful to make sure it is doing what you want it to do.

We call the `.calc()` method on an `Atoms` or `Structure` object to launch the calculation. There is also a `.calc_many()` option to parallelize calculations.


In [ ]:
relax_result = relax.calc(atoms)

The result is a dictionary with various fields you can query by.


In [ ]:
relax_result["energy"]

In [ ]:
relax_result["final_structure"]

We can convert the Pymatgen `Structure` back to `Atoms` if we'd like. Don't worry that the return type is `MSONAtoms`; it is effectively the same thing as a normal `Atoms` object.


In [ ]:
relax_result["final_structure"].to_ase_atoms()

## Gibbs Free Energy


We have to relax the Cu structure and then carry out a phonon mode calculation to get the Gibbs free energy. Conveniently, `matcalc` provides a single function to do both, and it will calculate the Gibbs free energy for us too. It does this in the `QHACalc` class, which is the quasi-harmonic approximation (QHA) method.


In [ ]:
from matcalc import QHACalc

qha = QHACalc(
    calc,
    t_min=0,  # K
    t_max=1000,  # K
    fmax=0.01,  # eV/A
    optimizer="BFGS",
    pressure=1e-4,  # GPa
    relax_structure=True,
)

In [ ]:
result = qha.calc(atoms)

In [ ]:
T_vals = result["temperatures"]
print(T_vals)

In [ ]:
G_vals = result["gibbs_free_energies"]
print(G_vals)

Now we can plot G vs. T if we'd like. Note that there is an off-by-one indexing issue that [will be resolved](https://github.com/materialyzeai/matcalc/issues/150) later.


In [ ]:
import matplotlib.pyplot as plt

plt.plot(T_vals[0:-1], G_vals)
plt.xlabel("T (K)")
plt.ylabel("G (eV)")